# 🛍️ Shopping Mall Customer Segmentation
## Step 0 — Exploratory Data Analysis (EDA) & Data Pre-processing (Optimized)
**Dataset:** Shopping_Mall_Customer_Segmentation_Data_.csv  

### 📌 老师反馈优化点：
1. **有意义的可视化**：不再只是展示图片，而是解释每张图对业务的意义（例如：性别与消费的关系）。
2. **Confusion Matrix**：在预处理阶段，如果存在分类标签（如性别），我们可以通过混淆矩阵观察特征与类别的关联性（虽然聚类是无监督的，但 EDA 阶段可以分析已知标签）。
3. **详细解释**：为每个步骤添加了“为什么做这个”和“结果说明”。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

print('✅ Libraries imported successfully!')

## 1. Load Dataset
**目的**：读取原始数据，了解数据的基本结构（行数、列数、数据类型）。

In [ ]:
df = pd.read_csv('data/Shopping Mall Customer Segmentation Data .csv')
print(f'Dataset Shape: {df.shape}')
df.head()

## 2. Data Cleaning & Encoding
**目的**：处理缺失值，并将分类变量（如 Gender）转换为数值，以便算法处理。

In [ ]:
# 检查缺失值
print("Missing values:\n", df.isnull().sum())

# 编码性别
le = LabelEncoder()
df['Gender_Code'] = le.fit_transform(df['Gender']) # Female=0, Male=1

# 删除不需要的 ID 列
df_numeric = df.drop(['Customer ID', 'Gender'], axis=1)
print("\n✅ Data cleaned and encoded.")

## 3. Meaningful EDA (有意义的数据分析)
### 3.1 Gender vs Spending Score (性别与消费分数的关联)
**意义**：了解不同性别的消费能力是否有显著差异。如果某一性别消费分数明显更高，商场可以针对性地进行营销。

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='Gender', y='Spending Score', data=df, palette='Set2')
plt.title('Spending Score Distribution by Gender')
plt.show()

print("💡 意义：通过箱线图，我们可以观察中位数和分布范围。如果男性的中位数高于女性，说明男性客户在该商场的平均消费意愿更强。")

### 3.2 Income vs Spending Score (收入与消费的关系)
**意义**：这是聚类最核心的两个维度。我们想看是否存在天然的群体（如：高收入高消费、高收入低消费）。

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Annual Income', y='Spending Score', hue='Gender', alpha=0.6)
plt.title('Annual Income vs Spending Score')
plt.show()

print("💡 意义：散点图可以直观展示客户的分布。如果看到明显的簇（Clusters），说明聚类算法会有很好的效果。")

## 4. Confusion Matrix in Preprocessing (混淆矩阵分析)
**意义**：老师建议做 Confusion Matrix。在预处理阶段，我们可以通过将连续变量（如 Spending Score）分桶，来观察它与性别（Gender）的匹配程度。这能帮我们理解特征之间是否存在某种“分类”逻辑。

In [ ]:
# 将 Spending Score 分为 'Low' 和 'High' 两类（以中位数为界）
df['Spending_Level'] = pd.qcut(df['Spending Score'], q=2, labels=['Low', 'High'])

# 计算性别与消费等级的混淆矩阵
cm = confusion_matrix(df['Gender'], df['Spending_Level'])

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Low Spending', 'High Spending'], 
            yticklabels=['Female', 'Male'])
plt.xlabel('Spending Level (Binned)')
plt.ylabel('Gender')
plt.title('Confusion Matrix: Gender vs Spending Level')
plt.show()

print("💡 意义：混淆矩阵展示了性别与消费水平的交叉分布。例如，如果左上角数字很大，说明女性大多属于低消费群体。这为后续聚类提供了业务背景。")

## 5. Feature Scaling & Saving
**目的**：将数据标准化（均值为0，方差为1），因为聚类算法（如 K-Means）对特征尺度非常敏感。

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_numeric)

# 保存预处理后的数据供后续 Notebook 使用
np.save('X_scaled.npy', X_scaled)
df_numeric.to_csv('data/data_preprocessed.csv', index=False)
print("✅ Preprocessed data saved as X_scaled.npy and data_preprocessed.csv")